# EC売上・顧客分析

架空のECサイトデータを使い、売上トレンド・カテゴリ別収益・顧客LTV・新規/既存比較・売上集中度を分析します。

---

**実行環境:** MySQL 8.0 / Python 3 / pandas  
**DB:** `sql_portfolio`（`SETUP.md` の手順で事前に構築）

## セットアップ

In [1]:
import mysql.connector
import pandas as pd
from IPython.display import display, HTML

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

con = mysql.connector.connect(
    host='localhost', user='root', password='',
    database='sql_portfolio',
    charset='utf8mb4'
)

def run(sql):
    return pd.read_sql(sql, con)


---

## 分析クエリ

### 01. 月次売上推移・前月比分析

**ビジネス課題:** 売上の月次トレンドと前月比を定量化し、異常値の早期検知や施策効果の評価に活用する。

**使用テクニック:** CTE, `LAG()`, `DATE_FORMAT`, `NULLIF`

In [2]:
sql = '''
-- Step 1: 月ごとの売上合計を算出（未決済・キャンセルは除外）
WITH monthly_sales AS (
    SELECT
        DATE_FORMAT(o.order_date, '%Y-%m') AS ym,
        SUM(oi.quantity * oi.unit_price)   AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status IN ('paid','shipped')
    GROUP BY DATE_FORMAT(o.order_date, '%Y-%m')
)
-- Step 2: LAG() で前月の売上を取得し、前月比（%）を計算
SELECT
    ym,
    revenue,
    LAG(revenue) OVER (ORDER BY ym) AS prev_revenue,
    ROUND(
        (revenue - LAG(revenue) OVER (ORDER BY ym)) /
        NULLIF(LAG(revenue) OVER (ORDER BY ym), 0) * 100,
        1
    ) AS mom_change_pct
FROM monthly_sales
ORDER BY ym;
'''
df = run(sql)
display(df)

C:\Users\willi\AppData\Local\Temp\ipykernel_23940\2420247800.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, con)


,ym,revenue,prev_revenue,mom_change_pct
0,2024-01,"44,200.00",NaN,NaN
1,2024-02,"40,360.00","44,200.00",-8.70
2,2024-03,"32,200.00","40,360.00",-20.20
3,2024-04,"31,800.00","32,200.00",-1.20
4,2024-05,"34,400.00","31,800.00",8.20
5,2024-06,"65,260.00","34,400.00",89.70
6,2024-07,"61,400.00","65,260.00",-5.90
7,2024-08,"30,060.00","61,400.00",-51.00
8,2024-09,"21,500.00","30,060.00",-28.50
9,2024-10,"70,580.00","21,500.00",228.30


### 02. カテゴリ別売上ランキング

**ビジネス課題:** 売上貢献度の高い商品カテゴリを特定し、仕入れやプロモーション戦略の優先順位付けに活用する。

**使用テクニック:** 4テーブル JOIN, `GROUP BY`, `ORDER BY DESC`

In [3]:
sql = '''
-- 注文明細 → 注文 → 商品 → カテゴリ の4テーブルを結合し、
-- カテゴリ単位で売上を集計してランキング化
SELECT
    c.category_name,
    SUM(oi.quantity * oi.unit_price) AS revenue
FROM order_items oi
JOIN orders    o ON oi.order_id   = o.order_id
JOIN products  p ON oi.product_id = p.product_id
JOIN categories c ON p.category_id = c.category_id
WHERE o.status IN ('paid','shipped')
GROUP BY c.category_id, c.category_name
ORDER BY revenue DESC
LIMIT 10;
'''
df = run(sql)
display(df)

C:\Users\willi\AppData\Local\Temp\ipykernel_23940\2420247800.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, con)


,category_name,revenue
0,Home & Kitchen,"175,460.00"
1,Clothing,"136,800.00"
2,Electronics,"106,400.00"
3,Books,"77,400.00"
4,Food,"24,200.00"


### 03. 顧客LTV（生涯価値）分析

**ビジネス課題:** 顧客ごとの累計売上・注文回数・平均注文額を算出し、VIP顧客の識別に活用する。

**使用テクニック:** CTE, JOIN, 集計関数

In [4]:
sql = '''
-- Step 1: 注文単位で小計を算出（集計粒度を注文→顧客に上げる前処理）
WITH order_amounts AS (
    SELECT
        o.order_id,
        o.customer_id,
        SUM(oi.quantity * oi.unit_price) AS order_amount
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status IN ('paid','shipped')
    GROUP BY o.order_id, o.customer_id
)
-- Step 2: 顧客単位で集約し、LTV・注文回数・平均注文額を算出
SELECT
    c.customer_id,
    c.customer_name,
    COUNT(oa.order_id)              AS order_count,
    SUM(oa.order_amount)            AS lifetime_value,
    ROUND(AVG(oa.order_amount), 2)  AS avg_order_value
FROM customers c
JOIN order_amounts oa ON c.customer_id = oa.customer_id
GROUP BY c.customer_id, c.customer_name
ORDER BY lifetime_value DESC
LIMIT 50;
'''
df = run(sql)
display(df)

C:\Users\willi\AppData\Local\Temp\ipykernel_23940\2420247800.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, con)


,customer_id,customer_name,order_count,lifetime_value,avg_order_value
0,29,小林誠,6,"59,900.00","9,983.33"
1,1,田中太郎,3,"57,500.00","19,166.67"
2,5,高橋翔太,5,"50,100.00","10,020.00"
3,13,松本健二,4,"43,400.00","10,850.00"
4,4,山田大輔,3,"42,500.00","14,166.67"
5,17,清水拓也,3,"23,800.00","7,933.33"
6,27,伊藤遥,4,"22,100.00","5,525.00"
7,25,高橋翔太,2,"22,000.00","11,000.00"
8,8,中村和也,3,"21,960.00","7,320.00"
9,20,前田浩,4,"21,560.00","5,390.00"


### 04. 新規顧客 vs 既存顧客の月次売上比較

**ビジネス課題:** 月ごとの売上が新規獲得とリピートのどちらに依存しているかを把握する。

**使用テクニック:** CTE, `CASE`式, ウィンドウ関数

In [5]:
sql = '''
-- Step 1: 顧客ごとの初回注文日を特定
WITH first_orders AS (
    SELECT
        customer_id,
        MIN(order_date) AS first_order_date
    FROM orders
    WHERE status IN ('paid','shipped')
    GROUP BY customer_id
),
-- Step 2: 各注文を「初回注文日と同日 → new」「それ以降 → existing」に分類
order_with_flag AS (
    SELECT
        o.*,
        CASE
            WHEN DATE(o.order_date) = DATE(f.first_order_date)
                THEN 'new'
            ELSE 'existing'
        END AS customer_type
    FROM orders o
    JOIN first_orders f ON o.customer_id = f.customer_id
    WHERE o.status IN ('paid','shipped')
)
-- Step 3: 月×顧客タイプ別に売上を集計
SELECT
    DATE_FORMAT(o.order_date, '%Y-%m') AS ym,
    customer_type,
    SUM(oi.quantity * oi.unit_price)   AS revenue
FROM order_with_flag o
JOIN order_items oi ON o.order_id = oi.order_id
GROUP BY DATE_FORMAT(o.order_date, '%Y-%m'), customer_type
ORDER BY ym, customer_type;
'''
df = run(sql)
display(df)

C:\Users\willi\AppData\Local\Temp\ipykernel_23940\2420247800.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, con)


,ym,customer_type,revenue
0,2024-01,new,"44,200.00"
1,2024-02,existing,"16,400.00"
2,2024-02,new,"23,960.00"
3,2024-03,existing,"11,400.00"
4,2024-03,new,"20,800.00"
5,2024-04,existing,"15,200.00"
6,2024-04,new,"16,600.00"
7,2024-05,existing,"15,800.00"
8,2024-05,new,"18,600.00"
9,2024-06,existing,"53,560.00"


### 05. 売上トップ10%顧客の集中度分析

**ビジネス課題:** 売上上位顧客への依存度を定量化し、顧客基盤のリスク評価に活用する。

**使用テクニック:** CTE(3段), `NTILE()`

In [6]:
sql = '''
-- Step 1: 顧客ごとの累計売上を算出
WITH customer_ltv AS (
    SELECT
        o.customer_id,
        SUM(oi.quantity * oi.unit_price) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status IN ('paid','shipped')
    GROUP BY o.customer_id
),
-- Step 2: NTILE(10) で顧客を売上順に10分割（デシル）
ranked AS (
    SELECT
        customer_id,
        revenue,
        NTILE(10) OVER (ORDER BY revenue DESC) AS decile
    FROM customer_ltv
),
-- Step 3: 上位10%（decile = 1）の売上合計と全体売上を集計
agg AS (
    SELECT
        SUM(revenue) AS total_revenue,
        SUM(CASE WHEN decile = 1 THEN revenue ELSE 0 END) AS top10_revenue
    FROM ranked
)
SELECT
    total_revenue,
    top10_revenue,
    ROUND(top10_revenue / total_revenue * 100, 1) AS top10_share_pct
FROM agg;
'''
df = run(sql)
display(df)

C:\Users\willi\AppData\Local\Temp\ipykernel_23940\2420247800.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, con)


,total_revenue,top10_revenue,top10_share_pct
0,"520,260.00","167,500.00",32.20


---

In [7]:
con.close()
print('接続を閉じました。')

接続を閉じました。
